In [37]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping

import functions
import pandas as pd

In [38]:
X_train, X_test, y_test = functions.load_SMD(2,2)

In [39]:
print("X_train.shape:", X_train.shape)
print("X_test.shape: ", X_test.shape)
print("y_test.shape: ", y_test.shape)

X_train.shape: (23699, 38)
X_test.shape:  (23700, 38)
y_test.shape:  (23700, 1)


In [40]:
timesteps = 5
n_features = X_train.shape[1]

# Encoder
inputs = Input(shape=(timesteps, n_features))
encoded = LSTM(64, return_sequences=False)(inputs)
latent = RepeatVector(timesteps)(encoded)

# Decoder
decoded = LSTM(64, return_sequences=True)(latent)
outputs = TimeDistributed(Dense(n_features))(decoded)

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse')

In [41]:
timewindows = tf.signal.frame(tf.convert_to_tensor(X_train, dtype=tf.float32), frame_length=timesteps, frame_step=1, axis=0)
testwindows = tf.signal.frame(tf.convert_to_tensor(X_test, dtype=tf.float32), frame_length=timesteps, frame_step=1, axis=0)

In [42]:
np.ravel(y_test)

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [44]:
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

model.fit(
    timewindows, timewindows,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[es]
)

# 4) Compute reconstruction errors on train set, to pick a threshold
X_train_pred = model.predict(timewindows)
train_mse = np.mean(np.power(timewindows - X_train_pred, 2), axis=(1, 2))
# e.g. choose threshold as the 95th percentile of train errors
threshold = np.percentile(train_mse, 99)
print(f"Reconstruction error threshold: {threshold:.4f}")

# 5) Compute reconstruction errors on test set
X_test_pred = model.predict(testwindows)
test_mse = np.mean(np.power(testwindows - X_test_pred, 2), axis=(1, 2))

y_scores = tf.Variable(tf.zeros_like(y_test, dtype=tf.float32))
for i, score in enumerate(test_mse):
    start_t = i
    end_t = i+timesteps
    y_scores[start_t: end_t].assign(tf.maximum(y_scores[start_t:end_t], score))

# 6) Derive binary predictions
y_pred = (y_scores.numpy() > threshold).astype(int)

# 7) Evaluate
print(classification_report(y_test, y_pred, target_names=['normal','anomaly']))
functions.display_metrics(y_pred, y_test)


Epoch 1/50
667/667 [==============================] - 4s 6ms/step - loss: 2.2257e-04 - val_loss: 2.6732e-04
Epoch 2/50
667/667 [==============================] - 4s 6ms/step - loss: 1.9783e-04 - val_loss: 2.3994e-04
Epoch 3/50
667/667 [==============================] - 4s 6ms/step - loss: 1.8192e-04 - val_loss: 2.2367e-04
Epoch 4/50
667/667 [==============================] - 4s 5ms/step - loss: 1.7086e-04 - val_loss: 2.2221e-04
Epoch 5/50
667/667 [==============================] - 4s 6ms/step - loss: 1.6257e-04 - val_loss: 2.1959e-04
Epoch 6/50
667/667 [==============================] - 4s 6ms/step - loss: 1.5518e-04 - val_loss: 1.9319e-04
Epoch 7/50
667/667 [==============================] - 4s 6ms/step - loss: 1.4769e-04 - val_loss: 1.9288e-04
Epoch 8/50
667/667 [==============================] - 4s 6ms/step - loss: 1.4137e-04 - val_loss: 1.7690e-04
Epoch 9/50
667/667 [==============================] - 4s 6ms/step - loss: 1.3568e-04 - val_loss: 1.7080e-04
Epoch 10/50
667/667 [=======